# HyperParams

### Importing Libraries

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [3]:
pip install bayesian-optimization #Only needed for the first time

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
  Created wheel for bayesian-optimization: filename=bayesian_optimization-1.2.0-py3-none-any.whl size=11685 sha256=8ac6312e4b0d67bcd1a0a071a50b7e512e3f7fee4d3f2c6cf4e8e4967174ce1c
  Stored in directory: /root/.cache/pip/wheels/fd/9b/71/f127d694e02eb40bcf18c7ae9613b88a6be4470f57a8528c5b
Successfully built bayesian-optimization


In [4]:
from bayes_opt import BayesianOptimization
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier as RFC

### Reading The Dataset

In [5]:
data = pd.read_csv("Telecom_data.csv")
data.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Preprocessing The Dataset

#### Removing The Null or Zero Values In The Dataset wrt TotalCharges

In [6]:
data["TotalCharges"] = data["TotalCharges"].apply(lambda x: float("0"+x.strip()))

#### Checking For Null Values In The Dataset

In [7]:
data.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

#### Converting The String Values To Binary

In [8]:
cols = ["gender", "Partner", "Dependents", "PhoneService", "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies", "Contract", "PaperlessBilling", "PaymentMethod", ]
lbl_enc = LabelEncoder()
for col in cols:
    data[col] = lbl_enc.fit_transform(data[col])
data.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,0,0,1,0,1,0,1,0,0,...,0,0,0,0,0,1,2,29.85,29.85,No
1,5575-GNVDE,1,0,0,0,34,1,0,0,2,...,2,0,0,0,1,0,3,56.95,1889.50,No
2,3668-QPYBK,1,0,0,0,2,1,0,0,2,...,0,0,0,0,0,1,3,53.85,108.15,Yes
3,7795-CFOCW,1,0,0,0,45,0,1,0,2,...,2,2,0,0,1,0,0,42.30,1840.75,No
4,9237-HQITU,0,0,0,0,2,1,0,1,0,...,0,0,0,0,0,1,2,70.70,151.65,Yes


#### Converting Churn Values To Binary

In [9]:
map_dict = {"Yes":1, "No":0}
data["Churn"] = data["Churn"].map(map_dict)
data["Churn"].value_counts()

cols_to_use = [col for col in data.columns if col not in ["customerID", "Churn"]]
x = data[cols_to_use].values
y = data["Churn"].values
data.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,0,0,1,0,1,0,1,0,0,...,0,0,0,0,0,1,2,29.85,29.85,0
1,5575-GNVDE,1,0,0,0,34,1,0,0,2,...,2,0,0,0,1,0,3,56.95,1889.50,0
2,3668-QPYBK,1,0,0,0,2,1,0,0,2,...,0,0,0,0,0,1,3,53.85,108.15,1
3,7795-CFOCW,1,0,0,0,45,0,1,0,2,...,2,2,0,0,1,0,0,42.30,1840.75,0
4,9237-HQITU,0,0,0,0,2,1,0,1,0,...,0,0,0,0,0,1,2,70.70,151.65,1


### Spliting The Dataset For Training And Testing

In [10]:
train_X, test_X, train_y, test_y = train_test_split(x, y, test_size=0.2, random_state=2020)

### Bayesian Optimization

#### Optimization Function

In [11]:
def rfc_cv(n_estimators, min_samples_split, max_features, data, targets):
    estimator = RFC(
        n_estimators=n_estimators,
        min_samples_split=min_samples_split,
        max_features=max_features,
        random_state=2
    )
    cval = cross_val_score(estimator, data, targets, scoring='neg_log_loss', cv=4)
    return cval.mean()

def optimize_rfc(data, targets):
    def rfc_crossval(n_estimators, min_samples_split, max_features):
        return rfc_cv(
            n_estimators=int(n_estimators),
            min_samples_split=int(min_samples_split),
            max_features=max(min(max_features, 0.999), 1e-3),
            data=data,
            targets=targets,
        )
    optimizer = BayesianOptimization(
        f=rfc_crossval,
        pbounds={
            "n_estimators": (10, 250),
            "min_samples_split": (2, 25),
            "max_features": (0.1, 0.999),
        },
        random_state=1234,
        verbose=2
    )
    optimizer.maximize(n_iter=10)

    print("Final result:", optimizer.max)
    
    return optimizer
    
result = optimize_rfc(train_X, train_y)

|   iter    |  target   | max_fe... | min_sa... | n_esti... |
-------------------------------------------------------------
|  1        | -0.4396   |  0.2722   |  16.31    |  115.1    |
|  2        | -0.4545   |  0.806    |  19.94    |  75.42    |
|  3        | -0.4405   |  0.3485   |  20.44    |  240.0    |
|  4        | -0.4816   |  0.8875   |  10.23    |  130.2    |
|  5        | -0.4632   |  0.7144   |  18.39    |  98.86    |
|  6        | -0.4588   |  0.8021   |  18.14    |  230.7    |
|  7        | -0.4721   |  0.9241   |  16.19    |  80.11    |
|  8        | -0.4482   |  0.8985   |  23.55    |  73.52    |
|  9        | -0.4546   |  0.8987   |  21.88    |  114.4    |
|  10       | -0.4307   |  0.1      |  12.0     |  112.6    |
|  11       | -0.51     |  0.999    |  4.887    |  111.2    |
|  12       | -0.4304   |  0.1      |  14.92    |  111.2    |
|  13       | -0.4492   |  0.626    |  21.32    |  246.0    |
|  14       | -0.4622   |  0.7528   |  14.3     |  241.8    |
|  15   

#### The Best Set Of Parameters And Values

In [13]:
result.max

{'target': -0.42985013588025345,
 'params': {'max_features': 0.1,
  'min_samples_split': 25.0,
  'n_estimators': 237.63878480801307}}

In [3]:
from tqdm import tqdm
import time

# Example loading bar
for i in tqdm(range(10), desc="Loading..."):
    time.sleep(0.05)  
    tasks = ["Initializing AI agent", "Loading dataset", "Preprocessing data", "Training model", "Validating model", "Testing model", "Finalizing"]
    for task in tasks:
        print(f"{task}...")
        for _ in tqdm(range(50), desc=f"{task} Progress"):
            time.sleep(0.02)  
    print("All tasks completed successfully!")

Loading...:   0%|          | 0/10 [00:00<?, ?it/s]

Initializing AI agent...


Initializing AI agent Progress: 100%|██████████| 50/50 [00:01<00:00, 48.30it/s]


Initializing AI agent completed!

Loading dataset...


Loading dataset Progress: 100%|██████████| 50/50 [00:01<00:00, 48.35it/s]


...

Validating model completed!

Testing model...


Testing model Progress: 100%|██████████| 50/50 [00:01<00:00, 48.30it/s]


Testing model completed!

Finalizing...


Loading...: 100%|██████████| 10/10 [01:13<00:00,  7.32s/it]

Finalizing completed!

All tasks completed successfully!


In [ ]:
import os
import io
import json
import zipfile
import tempfile
import subprocess
from pathlib import Path
from datetime import datetime
from flask import Flask, request, jsonify
import requests
from git import Repo, GitCommandError
import shutil
import argparse

"""
GitSyncDoc API - cell to place in Jupyter or save as a file (e.g. git_sync_doc.py)

What it does:
- Pulls or clones a git repository.
- Collects code files, zips them.
- Uploads the zip (and metadata) to a configurable AI documentation endpoint.
- Requests a professional design + documentation artifact and saves outputs.

Requirements (install once):
pip install Flask GitPython requests python-magic

Environment variables / config (export or set in notebook):
- GIT_REPO_URL   : HTTPS or SSH URL of the repo to sync
- GIT_BRANCH     : branch to checkout/pull (default: main)
- LOCAL_REPO_DIR : local path to clone/update (default: ./synced_repo)
- AI_API_URL     : URL of your AI model endpoint that accepts file uploads / prompts
- AI_API_KEY     : API key / bearer token for the AI endpoint (if required)
- ALLOWED_EXTS   : comma-separated extensions to include (default: .py,.md,.ipynb,.yml,.yaml,.json)

How to run (example):
1) Export env vars:
    export GIT_REPO_URL="https://github.com/you/your-repo.git"
    export AI_API_URL="https://api.your-ml-service/v1/docs"
    export AI_API_KEY="sometoken"
2) Start the API:
    python git_sync_doc.py
3) Trigger update via HTTP:
    curl -X POST "http://127.0.0.1:5000/sync" -H "Content-Type: application/json" -d '{"force": true}'
4) Check outputs in ./outputs (design.md, docs.zip, metadata.json)

Notes:
- The script assumes the AI endpoint accepts multipart/form-data with fields:
     - "file": zip archive
     - "metadata": JSON string with repo info
     - "prompt": Optional textual instruction for doc generation
  Adjust upload logic in upload_to_model() to match your endpoint.

Security:
- For production, secure the Flask endpoint, restrict repo URLs, and validate inputs.
"""



# Optional: use GitPython if available; fallback to git CLI
try:
     HAVE_GITPY = True
except Exception:
     HAVE_GITPY = False

# ---- Configuration helpers ----
GIT_REPO_URL = os.environ.get("GIT_REPO_URL", "").strip()
GIT_BRANCH = os.environ.get("GIT_BRANCH", "main")
LOCAL_REPO_DIR = Path(os.environ.get("LOCAL_REPO_DIR", "./synced_repo"))
AI_API_URL = os.environ.get("AI_API_URL", "").strip()
AI_API_KEY = os.environ.get("AI_API_KEY", "").strip()
ALLOWED_EXTS = os.environ.get("ALLOWED_EXTS", ".py,.md,.ipynb,.yml,.yaml,.json")
OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXT_SET = set(e.strip().lower() for e in ALLOWED_EXTS.split(",") if e.strip())

app = Flask(__name__)

# ---- Git operations ----
def clone_or_update_repo(repo_url: str, branch: str, local_dir: Path, force_clone=False):
     local_dir = Path(local_dir)
     if local_dir.exists() and not force_clone:
          # try pull
          if HAVE_GITPY:
                try:
                     repo = Repo(str(local_dir))
                     origin = repo.remotes.origin
                     origin.fetch()
                     repo.git.checkout(branch)
                     origin.pull(branch)
                     return local_dir
                except Exception:
                     # fallback to cli
                     pass
          # CLI fallback
          try:
                subprocess.run(["git", "-C", str(local_dir), "fetch"], check=True)
                subprocess.run(["git", "-C", str(local_dir), "checkout", branch], check=True)
                subprocess.run(["git", "-C", str(local_dir), "pull", "origin", branch], check=True)
                return local_dir
          except subprocess.CalledProcessError as e:
                # if update fails, remove and reclone
                for _ in range(1):
                     pass
     # clone fresh
     if local_dir.exists():
          # remove directory to ensure clean clone
          shutil.rmtree(local_dir)
     local_dir.mkdir(parents=True, exist_ok=True)
     if HAVE_GITPY:
          Repo.clone_from(repo_url, str(local_dir), branch=branch)
     else:
          subprocess.run(["git", "clone", "--branch", branch, repo_url, str(local_dir)], check=True)
     return local_dir

# ---- File collection and zipping ----
def collect_files_to_zip(repo_dir: Path, ext_set=EXT_SET, exclude_dirs=("node_modules", ".git")):
     buf = io.BytesIO()
     with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as z:
          for root, dirs, files in os.walk(repo_dir):
                # filter dirs
                dirs[:] = [d for d in dirs if d not in exclude_dirs]
                for f in files:
                     if Path(f).suffix.lower() in ext_set:
                          full = Path(root) / f
                          arcname = full.relative_to(repo_dir)
                          z.write(full, arcname.as_posix())
     buf.seek(0)
     return buf

# ---- Metadata builder ----
def build_metadata(repo_dir: Path, repo_url: str, branch: str):
     # Basic metadata; extend as needed e.g., last commits, contributors
     metadata = {
          "repo_url": repo_url,
          "branch": branch,
          "collected_at": datetime.utcnow().isoformat() + "Z",
          "repo_path": str(repo_dir.resolve()),
     }
     # try to get last commit
     try:
          if HAVE_GITPY:
                repo = Repo(str(repo_dir))
                commit = repo.head.commit
                metadata.update({
                     "last_commit": str(commit.hexsha),
                     "last_commit_message": commit.message.strip(),
                     "last_commit_author": str(commit.author),
                })
          else:
                out = subprocess.check_output(["git", "-C", str(repo_dir), "rev-parse", "HEAD"]).decode().strip()
                metadata["last_commit"] = out
     except Exception:
          pass
     return metadata

# ---- Upload to AI model endpoint ----
def upload_to_model(zip_bytes_io: io.BytesIO, metadata: dict, prompt: str = None, api_url: str = None, api_key: str = None):
     api_url = api_url or AI_API_URL
     if not api_url:
          raise ValueError("AI_API_URL not configured")
     headers = {}
     if api_key:
          headers["Authorization"] = f"Bearer {api_key}"
     files = {
          "file": ("repo_snapshot.zip", zip_bytes_io, "application/zip"),
          "metadata": (None, json.dumps(metadata), "application/json"),
     }
     if prompt:
          files["prompt"] = (None, prompt)
     resp = requests.post(api_url, files=files, headers=headers, timeout=300)
     resp.raise_for_status()
     return resp.json() if resp.headers.get("content-type", "").startswith("application/json") else resp.content

# ---- Higher-level orchestration ----
def run_sync_and_document(force=False, prompt_override=None):
     if not GIT_REPO_URL:
          raise ValueError("GIT_REPO_URL not set")
     repo_dir = clone_or_update_repo(GIT_REPO_URL, GIT_BRANCH, LOCAL_REPO_DIR, force_clone=force)
     zip_buf = collect_files_to_zip(repo_dir)
     metadata = build_metadata(repo_dir, GIT_REPO_URL, GIT_BRANCH)
     # Compose a professional prompt (tweak as needed)
     prompt = prompt_override or (
          "You are a senior software engineer and technical writer. Produce:\n"
          "1) A professional Design Document describing architecture, modules, data flow, and dependencies.\n"
          "2) Developer Documentation (README-style) with setup, run, testing instructions, and code examples.\n"
          "3) A high-level file-by-file summary explaining purpose and key functions.\n"
          "Format your response as JSON with keys: design_md, readme_md, file_summaries (list of {path, summary})."
     )
     # Upload to model and get docs
     resp = upload_to_model(zip_buf, metadata, prompt, api_url=AI_API_URL, api_key=AI_API_KEY)
     # Save results
     timestamp = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
     meta_path = OUTPUT_DIR / f"metadata_{timestamp}.json"
     with open(meta_path, "w", encoding="utf-8") as f:
          json.dump(metadata, f, indent=2)
     # If API returned JSON with docs, save them
     if isinstance(resp, dict):
          if "design_md" in resp:
                (OUTPUT_DIR / f"design_{timestamp}.md").write_text(resp["design_md"], encoding="utf-8")
          if "readme_md" in resp:
                (OUTPUT_DIR / f"readme_{timestamp}.md").write_text(resp["readme_md"], encoding="utf-8")
          if "file_summaries" in resp:
                (OUTPUT_DIR / f"file_summaries_{timestamp}.json").write_text(json.dumps(resp["file_summaries"], indent=2), encoding="utf-8")
     else:
          # If a binary (zip) returned, save as docs.zip
          out_path = OUTPUT_DIR / f"docs_{timestamp}.zip"
          with open(out_path, "wb") as f:
                f.write(resp)
     # Always save the uploaded snapshot
     with open(OUTPUT_DIR / f"repo_snapshot_{timestamp}.zip", "wb") as f:
          f.write(zip_buf.getbuffer())
     return {"status": "ok", "metadata_file": str(meta_path)}

# ---- Flask endpoints ----
@app.route("/sync", methods=["POST"])
def sync_endpoint():
     data = request.get_json(silent=True) or {}
     force = bool(data.get("force", False))
     prompt = data.get("prompt")
     try:
          result = run_sync_and_document(force=force, prompt_override=prompt)
          return jsonify({"ok": True, "result": result})
     except Exception as e:
          return jsonify({"ok": False, "error": str(e)}), 500

@app.route("/status", methods=["GET"])
def status():
     return jsonify({
          "ok": True,
          "repo_url": GIT_REPO_URL,
          "branch": GIT_BRANCH,
          "local_dir": str(LOCAL_REPO_DIR.resolve()),
          "outputs": [str(p) for p in OUTPUT_DIR.iterdir()],
     })

# ---- CLI support when run directly ----
if __name__ == "__main__":
     p = argparse.ArgumentParser()
     p.add_argument("--force", action="store_true", help="Force fresh clone")
     p.add_argument("--prompt", type=str, help="Override default AI prompt")
     p.add_argument("--serve", action="store_true", help="Run Flask server instead of one-shot")
     args = p.parse_args()
     if args.serve:
          app.run(host="0.0.0.0", port=5000, debug=False)
     else:
          out = run_sync_and_document(force=args.force, prompt_override=args.prompt)
          print("Done. Outputs written to:", OUTPUT_DIR)
          print(json.dumps(out, indent=2))

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, log_loss, confusion_matrix
import seaborn as sns
import joblib

# Use the Bayesian optimization result to train a final RFC and evaluate on test set

# Imports needed for evaluation / saving (not imported earlier)
import matplotlib.pyplot as plt

# Ensure optimizer result exists
if 'result' not in globals() or result is None:
    raise RuntimeError("No optimization result found. Run the Bayesian optimization cell first (cell 20).")

# Extract and convert best params
best = result.max["params"]
n_estimators = int(best["n_estimators"])
min_samples_split = int(best["min_samples_split"])
max_features = float(best["max_features"])
max_features = max(min(max_features, 0.999), 1e-3)  # clip to safe range

print("Best params (raw):", best)
print("Best params (converted):", {"n_estimators": n_estimators, "min_samples_split": min_samples_split, "max_features": max_features})
print("Best target (CV neg_log_loss):", result.max["target"])

# Train final RandomForestClassifier on full train split
final_clf = RFC(
    n_estimators=n_estimators,
    min_samples_split=min_samples_split,
    max_features=max_features,
    random_state=42,
    n_jobs=-1
)
final_clf.fit(train_X, train_y)

# Predict & evaluate
probs = final_clf.predict_proba(test_X)[:, 1]
preds = final_clf.predict(test_X)

metrics = {
    "accuracy": accuracy_score(test_y, preds),
    "precision": precision_score(test_y, preds, zero_division=0),
    "recall": recall_score(test_y, preds, zero_division=0),
    "f1": f1_score(test_y, preds, zero_division=0),
    "roc_auc": roc_auc_score(test_y, probs),
    "log_loss": log_loss(test_y, probs),
}
print("\nTest set metrics:")
for k, v in metrics.items():
    print(f" - {k}: {v:.4f}")

# Confusion matrix
cm = confusion_matrix(test_y, preds)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# Feature importances (use cols_to_use defined earlier)
if 'cols_to_use' in globals():
    feat_names = list(cols_to_use)
else:
    feat_names = [f"f_{i}" for i in range(train_X.shape[1])]

importances = final_clf.feature_importances_
idx_sorted = importances.argsort()[::-1]
top_k = 20
top_idx = idx_sorted[:top_k]

plt.figure(figsize=(8, min(0.3 * top_k, 10)))
sns.barplot(x=importances[top_idx], y=[feat_names[i] for i in top_idx])
plt.title("Top Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

# Print top features
print("\nTop features:")
for i in top_idx[:top_k]:
    print(f" - {feat_names[i]}: {importances[i]:.4f}")

# Save the trained model
joblib.dump(final_clf, "rfc_best_model.joblib")
print("\nModel saved to: rfc_best_model.joblib")

# Show top optimization candidates history (if available)
if hasattr(result, "res"):
    try:
        history = pd.DataFrame([{"target": r["target"], **r["params"]} for r in result.res])
        print("\nTop 5 candidates from optimization history (sorted by target desc):")
        display(history.sort_values("target", ascending=False).head(5))
    except Exception:
        pass